# Large language models

> From next-token prediction to something that answers your question: tokenisation, scaling laws, the three training stages, and exactly what temperature does.

Read this chapter at `/learn/14-llms/`. Exported from `src/content/chapters/14-llms.mdx` — edit there, not here.


A language model does exactly one thing: given some text, predict what comes
next.

That's the whole job description. Every capability you've ever seen from one —
answering, translating, refactoring, summarising, refusing — is that single
operation, scaled up and then shaped. I know that sounds like an oversimplification
made for teaching. It isn't. By the end of today you'll have built each piece.

## Tokens

Models don't see characters, and they don't see words. They see **tokens** —
produced by a compression algorithm fitted to a corpus.

Let's just build the algorithm. It's shorter than you'd think.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from collections import Counter

def train_bpe(text, n_merges=60):
    """Repeatedly merge the most frequent adjacent pair. That is the whole algorithm."""
    tokens = list(text)
    merges = []
    for _ in range(n_merges):
        pairs = Counter(zip(tokens, tokens[1:]))
        if not pairs:
            break
        best, count = pairs.most_common(1)[0]
        if count < 2:
            break
        merged, i = [], 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best:
                merged.append(tokens[i] + tokens[i + 1]); i += 2
            else:
                merged.append(tokens[i]); i += 1
        tokens, _ = merged, merges.append("".join(best))
    return tokens, merges

corpus = """the model predicts the next token, and the next token after that.
a model that predicts well has learned something about the text it read.
the objective is prediction; the capability is a side effect of the objective.
tokens are not words. tokens are whatever the compressor found worth naming.
a token is a unit of text that appeared often enough to deserve its own symbol.
the tokenizer is fitted to a corpus before the model is trained on that corpus.
prediction over trillions of tokens is expensive, and the expense is the point.
the model reads text, the model writes text, and nothing in between is magic.
what the model learns is a distribution over the next token given the previous.
learning a distribution is not the same as learning the truth of a sentence.
"""
tokens, merges = train_bpe(corpus)
print("first merges learned:", merges[:12])
print(f"\n{len(corpus)} characters -> {len(tokens)} tokens "
      f"({len(corpus) / len(tokens):.2f} chars per token)")

Look at the merges it learned, in order. `th`, then `he`, then `the`…

Now notice what *didn't* happen: nothing linguistic. Nobody consulted a
dictionary. The algorithm found that certain adjacent characters keep appearing
together and glued them.

And so *the*, *token* and *prediction* became tokens — **because they were
common, not because they are words**. That distinction is going to explain a
surprising amount.

In [ ]:
vocab = sorted(set(tokens), key=len, reverse=True)
print("longest learned tokens:", vocab[:8])
print("\nEnglish is ~4 chars/token. Consequences:")
for label, chars in [("English prose", 4.0), ("code (whitespace)", 2.8),
                     ("rare proper nouns", 1.6), ("non-Latin scripts", 1.2)]:
    print(f"  {label:20s} ~{chars:.1f} chars/token  -> {4.0 / chars:.1f}x the tokens per character")

This one design decision explains several things that otherwise look like
inexplicable stupidity.

**Why models are bad at spelling and arithmetic.** "strawberry" might be two or
three tokens. The model never sees the individual letters at all — so counting
the r's is genuinely, structurally hard for it, in the way that counting the
atoms in a brick is hard for you. Likewise `1234` might be one token and `1235`
another, with no structural relationship between them. Digit-by-digit
tokenisation, which newer models use, is a direct fix for exactly this.

**Why some languages cost more.** A vocabulary fitted mostly to English spends
more tokens per character on other scripts — so the same content costs more money
*and* eats more of the context window. That's a real equity problem and an active
area of work, not just a curiosity.

**Why token counts aren't word counts.** Roughly 0.75 words per token for
English, and considerably worse for code and structured data.

## The objective

In [ ]:
words = "the cat sat on the mat and the cat purred".split()
for i in range(1, 6):
    print(f"  context {' '.join(words[:i]):28s} -> predict '{words[i]}'")

That's it. That is the entire training signal for every language model on Earth.

Cross-entropy on the next token, over trillions of
tokens. And because the label is *part of the input*, no human labels anything,
and every document ever written becomes training data.

This is the self-supervision from
[Chapter 3](/learn/03-the-shape-of-problems/), and it's why this approach won:
**it made data free.**

In [ ]:
def build_bigram(text):
    ws = text.split()
    counts = {}
    for a, b in zip(ws, ws[1:]):
        counts.setdefault(a, Counter())[b] += 1
    return counts

bg = build_bigram(corpus)
print("after 'the', the distribution is:")
for w, c in bg["the"].most_common():
    print(f"   {w:14s} {c / sum(bg['the'].values()):.3f}")

That's a language model. A genuinely terrible one — its context is a single word —
but a language model.

And here's the useful framing: a transformer is *this*, with a context of a
hundred thousand tokens and a learned representation instead of a count table.
The objective is identical. Character for character, the same idea. Everything
between the bigram and GPT is capacity.

## Sampling: what temperature actually does

The model outputs a probability for every token in the vocabulary. Turning that
distribution into actual text is a **separate decision** — and it's the one you
control at the API.

In [ ]:
def softmax(z, T=1.0):
    z = np.asarray(z, float) / T
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

logits = [3.2, 2.9, 2.1, 0.4, -1.0]
labels = ["cat", "dog", "bird", "rock", "xylophone"]

print(f"{'token':12s}" + "".join(f"{f'T={t}':>10s}" for t in [0.2, 0.7, 1.0, 1.5]))
for i, lab in enumerate(labels):
    row = "".join(f"{softmax(logits, t)[i]:10.3f}" for t in [0.2, 0.7, 1.0, 1.5])
    print(f"{lab:12s}{row}")

Read across a row. Temperature divides the logits before
softmax — and that's *all* it does.

Low temperature exaggerates the differences and piles all the probability onto
the favourite. High temperature flattens things out and gives unlikely tokens a
real chance.

In [ ]:
plt.figure(figsize=(5.6, 3))
for t in [0.2, 0.7, 1.0, 2.0]:
    plt.plot(labels, softmax(logits, t), "o-", label=f"T={t}")
plt.ylabel("probability"); plt.legend(); plt.title("temperature reshapes, it does not re-rank")
plt.tight_layout()

Look at the title of that plot, because it's the thing people get wrong.

Temperature **never changes the ranking**. Only how sharply the model commits to
it. "cat" is the favourite at every temperature; the question is only whether it
gets 95% of the mass or 30%.

`T → 0` is greedy decoding — always the argmax, fully deterministic, and prone to
getting stuck in repetitive loops. `T = 1` is the model's actual learned
distribution, unmodified.

**Top-k** keeps only the k most likely tokens. **Top-p** (nucleus) keeps the
smallest set whose probabilities sum to p. Both exist to cut off the long tail —
tokens that are individually implausible but *collectively* probable, and which
is where most of the obviously-wrong output comes from.

In [ ]:
rng = np.random.default_rng(0)

def generate(counts, start, n=14, T=1.0, seed=0):
    r = np.random.default_rng(seed)
    out = [start]
    for _ in range(n):
        nxt = counts.get(out[-1])
        if not nxt:
            break
        opts, cts = zip(*nxt.items())
        p = softmax(np.log(np.array(cts, float)), T)
        out.append(opts[int(np.argmax(p))] if T < 0.05 else opts[r.choice(len(opts), p=p)])
    return " ".join(out)

print("greedy (T→0):", generate(bg, "the", T=0.01))
print("sampled (T=1):", generate(bg, "the", T=1.0))

The greedy version loops. Round and round.

That's not a bug in this toy — it's exactly why production decoding samples, and
why repetition penalties had to be invented. Always taking the single most likely
next word is a reliable way to end up going in circles, which is a slightly
uncomfortable observation about optimisation in general.

## Scaling laws

Now the finding that justified the money.

In [ ]:
# Illustrative power law of the shape reported by Kaplan et al. (2020)
compute = np.logspace(0, 9, 60)
loss = 2.0 + 12.0 * compute ** -0.08

plt.figure(figsize=(5.4, 3))
plt.loglog(compute, loss)
plt.xlabel("training compute (arbitrary units)"); plt.ylabel("test loss")
plt.title("a straight line on log-log axes, over 9 orders of magnitude")
plt.grid(alpha=.3, which="both"); plt.tight_layout()

Loss falls as a **power law** in compute, parameters and data. Which means a
straight line on log-log axes, holding over *nine orders of magnitude*.

I'd like to flag how strange that is. There was no reason to expect it. Most
things in engineering hit a wall, or a knee, or a regime change. This didn't — it
just kept going, in a straight line, for a factor of a billion.

The practical consequence is what changed the industry: **you can run small
experiments and predict the loss of a model a thousand times larger, before
spending the money.**

That's what turned frontier training from a gamble into a capital-allocation
decision. Somebody could put a number in a spreadsheet and defend it.

The **Chinchilla** result (Hoffmann et al., 2022) corrected an important error in
the original story.

Earlier practice made models as large as the budget allowed and then undertrained
them. Chinchilla showed the compute-optimal ratio is roughly **20 tokens per
parameter** — meaning most models of that era were far too big for the data they
were fed.

But the follow-on point is more useful still: **compute-optimal is not
deployment-optimal.**

If a model is going to serve billions of requests, it's worth training a *smaller*
model far past its compute-optimal point. You pay training once and inference
forever, so shifting cost from inference to training is nearly always a good
trade at scale.

That reasoning — not a new architecture — is why small, heavily-overtrained models
became the norm. An economics argument shaped the models you use every day.

## Three stages

Here's something that surprises people: a base model trained purely on next-token
prediction **does not answer questions.**

It continues text. Ask it a question and a perfectly plausible continuation is
three more questions — because that's what documents containing questions
actually look like. It isn't being unhelpful. It's doing precisely what it was
trained to do.

Turning that into an assistant takes two more stages.

<div class="table-scroll">

| Stage | Data | What it produces |
|---|---|---|
| **1. Pretraining** | trillions of tokens of raw text | a base model: knows the world, follows no instructions |
| **2. Supervised fine-tuning** | ~10⁴–10⁶ curated (instruction, response) pairs | a model that answers rather than continues |
| **3. Preference tuning** | human rankings of pairs of responses | a model tuned for helpfulness, tone and refusal |

</div>

And here's the ratio worth knowing. Stage 1 is essentially *all* of the compute,
and it's where the capability comes from. Stages 2 and 3 are comparatively tiny —
and they determine almost everything you actually *experience*: the format, the
tone, the willingness to say "I don't know".

Most of what people call a model's personality was added in the last 1% of the
work.

**RLHF** is the classical form of stage 3: train a reward model to predict which
of two responses a human preferred, then optimise the language model against it
with reinforcement learning. **DPO** gets a similar result by deriving a loss
that skips the separate reward model — simpler, and now more common.

The single most useful mental correction in this chapter:

**The base model already has the knowledge. Fine-tuning mostly changes behaviour,
not facts.**

This is why "let's fine-tune it on our documentation" disappoints so reliably.
You wanted retrieval and you bought a style transfer. The model comes back
sounding like your docs and still doesn't know what's in them.

If the goal is "answer using our documents", the right architecture is almost
always retrieval ([Chapter 12](/learn/12-embeddings-and-tabular/)): embed the
documents, find the relevant ones, put them in the prompt.

Fine-tune when you want a *format* or a *behaviour* the model doesn't have. Not
when you want it to know something.

## In-context learning

Now the genuinely unexpected part. Put a few examples in the prompt, and the
model does the task — with no gradient updates at all.

In [ ]:
prompt = """Translate to French:
sea otter -> loutre de mer
cheese -> fromage
plush giraffe ->"""
print(prompt)
print("\nNo weights changed. The pattern in the context is doing the work.")

Nobody designed this. Nobody built a "learn from examples in the prompt" module.
It **emerged** from scale, and it's a large part of why language models are
useful as general-purpose tools rather than as trained classifiers you have to
commission.

Current understanding points at **induction heads** — attention heads that look
back for a previous occurrence of the current token and copy whatever followed
it.

Which is precisely the "look at a specific earlier position" mechanism you built
by hand [yesterday](/learn/13-attention-and-transformers/), in exercise 2. You
constructed one with a couple of matrices and a large number.

Here's the part I find lovely. Researchers can watch these heads *form* during
training — and their appearance coincides with a sharp jump in the model's
in-context ability. Not a gradual improvement. A step.

So there's a moment, somewhere in the middle of a training run, where a handful
of attention heads quietly work out how to copy patterns — and the model becomes
able to learn from examples for the rest of its life without ever changing a
weight again.

Nobody put that there. It was the cheapest way to reduce next-token loss, and it
happened to be one of the most useful capabilities the technology has.

## What they cannot do

Worth being precise about, because these failure modes are **structural** rather
than incidental. They're not bugs anybody forgot to fix.

**They have no separate notion of truth.** The objective is plausibility, and a
fluent falsehood scores extremely well on plausibility. "Hallucination" isn't a
malfunction — it's the objective working exactly as specified. There is no
component that was supposed to check.

**Their knowledge has a cut-off and no timestamp.** The weights froze at training
time. Anything newer has to come through the context window, and the model has no
reliable sense of which of its beliefs are stale.

**Fixed computation per token.** A transformer does exactly the same amount of
work for "2+2" as for a hard proof. This is why chain-of-thought works: writing
out intermediate steps literally **buys more forward passes**. That's a
mechanical explanation for a technique that looks psychological, and I think it's
one of the most clarifying facts in this chapter.

**Context is bounded and quadratic.** See
[yesterday's table](/learn/13-attention-and-transformers/). Long context is
expensive, and models attend unevenly across it — things in the middle get less
reliable treatment than things at the ends.

The engineering consequence, and it's the one that matters if you're going to
build on these:

**Treat a model call like a request to an unreliable third-party service that
returns confident, well-formatted output regardless of whether it's right.**

You already know how to program against that. You've done it for years.

Validate the response. Constrain the output shape. Keep the arithmetic in code.
Retrieve rather than recall. And never, ever let it be the only check on
something expensive.

None of that is special AI engineering. It's just the ordinary discipline you'd
apply to any dependency you don't control.

**"Why does the BPE cell stop before 60 merges?"** Because it exits when no pair
occurs twice — there's nothing left worth merging on a corpus this small.
Exercise 1 explores exactly that, and it's the interesting part.

**"Temperature 0 isn't actually available in some APIs."** Right — many clamp to a
small positive value, because true argmax decoding can be implemented differently
on different hardware. "Deterministic" is more of an aspiration than a guarantee.

**"What's the difference between top-k and top-p?"** Top-k always keeps exactly k
tokens, whether the distribution is sharp or flat. Top-p keeps however many are
needed to reach p of the mass — so it adapts: few tokens when the model is
confident, more when it isn't. Top-p is generally the better default.

**"If the base model has the knowledge, why does fine-tuning ever help?"** Because
behaviour is genuinely learnable: output format, refusal style, domain tone,
tool-calling syntax. Those are real things worth buying. Just don't expect facts.

**"Is a bigram model really a language model?"** Yes, completely — it assigns a
probability to the next token given context. It's just that its context is one
word, which makes it useless. The objective is what defines the category, not the
quality.

In [ ]:
# 1. Run train_bpe with n_merges = 5, 20, 100. Plot compression ratio
#    against merge count. Where does it flatten, and why?
#
# 2. At what temperature does 'rock' (logit 0.4) first exceed 5% probability?
#    Find it by bisection.
#
# 3. Implement top-p sampling: keep the smallest set of tokens whose
#    probabilities sum to >= p, renormalise, sample from that.

print("replace me")

For 1, think about what has to be true for a merge to be worth doing at all —
then look at the `if count < 2: break` line in `train_bpe`.

For 3, `np.argsort` then `np.cumsum` then `np.searchsorted` does the whole job in
three lines.

In [ ]:
ratios = []
merge_counts = [0, 10, 25, 50, 100, 200, 400]
for m in merge_counts:
    t, _ = train_bpe(corpus, n_merges=m)
    ratios.append(len(corpus) / len(t))
plt.figure(figsize=(5, 2.8))
plt.plot(merge_counts, ratios, "o-")
plt.xlabel("merges"); plt.ylabel("chars per token"); plt.tight_layout()
print("compression:", [round(r, 2) for r in ratios])

It rises steeply and then stops dead. On this corpus it stops at 110 merges,
because the loop exits once no adjacent pair occurs twice. There is simply nothing
left worth merging.

The same shape holds at real scale, for the same reason: early merges capture
genuinely frequent pairs, later ones capture increasingly rare ones, and the
return per merge falls away.

That's why production vocabularies sit around 30k–200k tokens rather than
millions. And the cost side is worth spelling out, because it's concrete: past
the knee, each additional vocabulary entry buys almost no compression while
costing an entire row of the embedding table *and* an entire row of the output
projection. For a 4096-dimensional model that's about 32 KB of parameters per
token, forever, on every copy of the model ever shipped.

In [ ]:
lo, hi = 0.05, 20.0
for _ in range(50):
    mid = (lo + hi) / 2
    if softmax(logits, mid)[3] < 0.05: lo = mid
    else: hi = mid
print(f"'rock' crosses 5% at T = {hi:.3f}   (p = {softmax(logits, hi)[3]:.4f})")

def top_p_sample(logits, p=0.9, T=1.0, seed=0):
    probs = softmax(logits, T)
    order = np.argsort(probs)[::-1]
    cum = np.cumsum(probs[order])
    keep = order[:np.searchsorted(cum, p) + 1]
    renorm = probs[keep] / probs[keep].sum()
    return int(np.random.default_rng(seed).choice(keep, p=renorm)), keep

for p in [0.5, 0.9, 0.99]:
    _, keep = top_p_sample(logits, p=p)
    print(f"top-p {p}: keeps {len(keep)} tokens -> {[labels[i] for i in keep]}")

Now notice how strongly top-p interacts with temperature, because this catches
people out in production.

At high temperature the distribution flattens — so the nucleus *grows*, and
admits more of the tail. Which is precisely the moment you least wanted it to.
The two settings compound rather than balancing.

That's why APIs expose both, and why cranking both up aggressively usually
produces worse output than adjusting either one alone. If you're tuning
generation quality, move one knob at a time.

Tomorrow: the branches this path didn't take — and how to keep learning without
one.